# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide to load and explore the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library.

### Dataset Source
The dataset source is referenced via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print out basic metadata information
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's examine which **record sets** are present, their `@id`, and a summary of their fields.

In [ ]:
# List all record set @ids and their fields
record_sets = list(dataset.record_sets.values())
print(f"Found {len(record_sets)} record sets:")

for rs in record_sets:
    print(f"\nRecord set: @id = {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")

Let's inspect example records from the main data table record set. The principal data table is frequently the first `record_set` listed. Replace `<record_set_id>` below with the actual `@id` from the overview if you want to explore another.

In [ ]:
# Show up to 3 example records from the first record set
main_record_set_id = record_sets[0].id if record_sets else None
if main_record_set_id:
    print(f"\nExample records from record set {main_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nLoaded DataFrame for record set {rs_id} (rows: {len(df)}, columns: {list(df.columns)})")

# Show columns and preview the main data table DataFrame
if main_record_set_id:
    print(f"\nField columns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    print("\nPreview:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We will now:
- Select a numeric field for demonstration
- Filter for large values
- Normalize and display group statistics
- Group by a selected categorical field

> **Tip:** Use the printed field columns above and pick fields with numeric data (e.g., age, intervals) and a group/categorical field (e.g., sex, anatomical location). 

We'll auto-detect a likely numeric and group field for demonstration; you can edit these as needed.

In [ ]:
# Attempt to detect a numeric and group field for demonstration
numeric_field = None
group_field = None
df = dataframes[main_record_set_id]
if not df.empty:
    # Find float or int columns
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    # Find first object/categorical column
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break
else:
    print("No data in the main table.")

print(f"Selected numeric field: {numeric_field}")
print(f"Selected group field: {group_field}")

# EDA: Filter and transform numeric field
if numeric_field and not df[numeric_field].isnull().all():
    # Choose a threshold relative to the data
    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records where {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
    display(filtered_df.head())
    
    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group by group_field, if available
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field} (numeric means):")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships in the data (using detected fields above).

In [ ]:
# Visualize numeric field distribution
if numeric_field and not df[numeric_field].isnull().all():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=40, ha='right')
        plt.show()
else:
    print("No visualization possible: missing numeric or group field.")

## 6. Conclusion

* We loaded the FAIR² dataset via its Croissant schema and explored its structure using `mlcroissant`.
* We previewed its record sets and fields, referencing all entities by their `@id`.
* Tabular data are extracted and basic filtering, normalization, grouping, and visualization were demonstrated.
* For more advanced analysis, consult the Croissant metadata to select detailed fields by their `@id`, and extend the EDA as required.

**References**

- [FAIR² dataset Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
- [mlcroissant documentation](https://github.com/mlcommons/croissant)
